# LangGraph - Multi-Agent и Subgraphs

## Что изучим:

1. **Multi-Agent** - несколько агентов работают вместе
2. **Supervisor Pattern** - один агент управляет другими
3. **Subgraphs** - модульная архитектура графов

### Зачем это нужно?

**Проблема одного агента:**
- Слишком много инструментов — путается
- Сложные задачи — теряет контекст
- Разные роли — конфликт промптов

**Решение — команда агентов:**
- Каждый агент — специалист в своей области
- Supervisor распределяет задачи
- Модульность — легко добавлять/убирать агентов

### Java аналогия:

Multi-agent = Микросервисная архитектура:
- Supervisor = API Gateway / Orchestrator
- Агенты = Отдельные сервисы
- Subgraphs = Модули / Bounded Contexts

In [1]:
# Установка
!pip install -q langgraph langchain langchain-openai python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
print(f"API Key: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key: sk-***...


In [3]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

# LLM
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0
)

print("Imports готовы!")

Imports готовы!


---

## 1. Простой Multi-Agent: Handoff Pattern

### Идея:

Агенты передают управление друг другу напрямую.

```
User → Agent A → Agent B → Agent A → Response
```

### Пример: Исследователь + Писатель

- **Researcher** — ищет информацию
- **Writer** — пишет текст на основе найденного

In [ ]:
# Состояние для команды агентов
class TeamState(TypedDict):
    messages: Annotated[list, operator.add]  # История сообщений
    next_agent: str  # Кто следующий
    research_data: str  # Данные от researcher
    final_output: str  # Финальный результат

print("State определён!")

In [ ]:
# Агент-исследователь
def researcher_agent(state: TeamState) -> TeamState:
    print("\n🔍 RESEARCHER работает...")
    
    messages = [
        SystemMessage(content="""Ты — исследователь. Твоя задача:
1. Проанализировать запрос пользователя
2. Найти/сгенерировать ключевые факты по теме
3. Вернуть структурированные данные для писателя

Формат ответа:
ФАКТЫ:
- факт 1
- факт 2
- факт 3
"""),
        HumanMessage(content=state["messages"][-1].content if state["messages"] else "")
    ]
    
    response = llm.invoke(messages)
    print(f"   Найдено: {response.content[:100]}...")
    
    return {
        "messages": [AIMessage(content=f"[Researcher]: {response.content}")],
        "research_data": response.content,
        "next_agent": "writer"
    }

print("Researcher создан!")

In [ ]:
# Агент-писатель
def writer_agent(state: TeamState) -> TeamState:
    print("\n✍️ WRITER работает...")
    
    messages = [
        SystemMessage(content="""Ты — профессиональный писатель. Твоя задача:
1. Взять данные от исследователя
2. Написать engaging текст на их основе
3. Сделать текст интересным и читаемым

Пиши кратко, по делу, с примерами.
"""),
        HumanMessage(content=f"""Данные от исследователя:
{state['research_data']}

Напиши на основе этого короткую статью (3-4 абзаца).""")
    ]
    
    response = llm.invoke(messages)
    print(f"   Написано: {response.content[:100]}...")
    
    return {
        "messages": [AIMessage(content=f"[Writer]: {response.content}")],
        "final_output": response.content,
        "next_agent": "end"
    }

print("Writer создан!")

In [ ]:
# Роутер — кто следующий?
def router(state: TeamState) -> Literal["researcher", "writer", "__end__"]:
    next_agent = state.get("next_agent", "researcher")
    
    if next_agent == "end":
        return "__end__"
    return next_agent

print("Router создан!")

In [ ]:
# Собираем граф
workflow = StateGraph(TeamState)

# Добавляем узлы
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)

# Входная точка — сразу идём в router
workflow.add_conditional_edges(
    START,
    router,
    {
        "researcher": "researcher",
        "writer": "writer",
        "__end__": END
    }
)

# После каждого агента — снова в router
workflow.add_conditional_edges("researcher", router)
workflow.add_conditional_edges("writer", router)

# Компилируем
app = workflow.compile()

print("Граф собран!")

In [ ]:
# Визуализация
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except:
    print(app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
result = app.invoke({
    "messages": [HumanMessage(content="Расскажи про квантовые компьютеры")],
    "next_agent": "researcher",
    "research_data": "",
    "final_output": ""
})

print("\n" + "="*50)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ:")
print("="*50)
print(result["final_output"])

### Что произошло:

1. User → **Researcher** (собрал факты)
2. Researcher → **Writer** (написал статью)
3. Writer → **END** (готово)

### Проблемы этого подхода:

- Жёсткий порядок: researcher → writer
- Нет гибкости в маршрутизации
- Сложно добавлять новых агентов

**Решение → Supervisor Pattern!**

---

## 2. Supervisor Pattern

### Идея:

Один "босс" решает, кого вызвать следующим.

```
User → Supervisor → Agent A
                 → Agent B  
                 → Agent C
                 → Response
```

### Java аналогия:

Supervisor = Orchestrator / Saga Coordinator
- Знает о всех сервисах
- Решает порядок вызовов
- Обрабатывает ошибки

In [ ]:
from langchain_core.tools import tool

# Определяем "работников"
WORKERS = ["researcher", "writer", "critic"]

# Состояние с supervisor
class SupervisorState(TypedDict):
    messages: Annotated[list, operator.add]
    next_worker: str
    research_data: str
    draft: str
    feedback: str

print("State для Supervisor готов!")

In [ ]:
# Supervisor — решает кого вызвать
def supervisor_node(state: SupervisorState) -> SupervisorState:
    print("\n👔 SUPERVISOR думает...")
    
    # Собираем контекст
    context = f"""
Текущее состояние:
- Research данные: {'есть' if state.get('research_data') else 'нет'}
- Черновик: {'есть' if state.get('draft') else 'нет'}
- Фидбек: {'есть' if state.get('feedback') else 'нет'}

Последние сообщения:
{[m.content[:100] for m in state['messages'][-3:]]}
"""
    
    messages = [
        SystemMessage(content=f"""Ты — supervisor команды из 3 работников:
- researcher: ищет информацию, собирает факты
- writer: пишет тексты на основе данных
- critic: проверяет качество, даёт фидбек

Твоя задача — решить, кого вызвать следующим.

Правила:
1. Сначала researcher (если нет данных)
2. Потом writer (если нет черновика)
3. Потом critic (если нет фидбека)
4. Если всё готово — FINISH

Ответь ОДНИМ словом: researcher, writer, critic, или FINISH
"""),
        HumanMessage(content=context)
    ]
    
    response = llm.invoke(messages)
    next_worker = response.content.strip().lower()
    
    # Валидация
    if next_worker not in WORKERS + ["finish"]:
        # Fallback логика
        if not state.get('research_data'):
            next_worker = "researcher"
        elif not state.get('draft'):
            next_worker = "writer"
        elif not state.get('feedback'):
            next_worker = "critic"
        else:
            next_worker = "finish"
    
    print(f"   Решение: {next_worker}")
    
    return {
        "messages": [AIMessage(content=f"[Supervisor]: Вызываю {next_worker}")],
        "next_worker": next_worker
    }

print("Supervisor создан!")

In [ ]:
# Работники
def researcher_node(state: SupervisorState) -> SupervisorState:
    print("\n🔍 RESEARCHER работает...")
    
    task = state["messages"][0].content  # Оригинальный запрос
    
    messages = [
        SystemMessage(content="Ты исследователь. Найди 3-5 ключевых фактов по теме. Кратко, по пунктам."),
        HumanMessage(content=task)
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Researcher]: {response.content}")],
        "research_data": response.content
    }

def writer_node(state: SupervisorState) -> SupervisorState:
    print("\n✍️ WRITER работает...")
    
    messages = [
        SystemMessage(content="Ты писатель. Напиши короткую статью (2-3 абзаца) на основе данных."),
        HumanMessage(content=f"Данные:\n{state['research_data']}")
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Writer]: {response.content}")],
        "draft": response.content
    }

def critic_node(state: SupervisorState) -> SupervisorState:
    print("\n🧐 CRITIC работает...")
    
    messages = [
        SystemMessage(content="""Ты критик. Оцени текст:
1. Что хорошо?
2. Что можно улучшить?
3. Общая оценка (1-10)

Будь конструктивен и краток."""),
        HumanMessage(content=f"Текст для проверки:\n{state['draft']}")
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Critic]: {response.content}")],
        "feedback": response.content
    }

print("Все работники созданы!")

In [ ]:
# Роутер от supervisor
def supervisor_router(state: SupervisorState) -> str:
    next_worker = state.get("next_worker", "researcher")
    
    if next_worker == "finish":
        return "__end__"
    return next_worker

print("Router готов!")

In [ ]:
# Собираем граф
workflow = StateGraph(SupervisorState)

# Узлы
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)

# Входная точка — supervisor
workflow.set_entry_point("supervisor")

# Supervisor решает куда идти
workflow.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "researcher": "researcher",
        "writer": "writer",
        "critic": "critic",
        "__end__": END
    }
)

# После каждого работника — обратно к supervisor
workflow.add_edge("researcher", "supervisor")
workflow.add_edge("writer", "supervisor")
workflow.add_edge("critic", "supervisor")

# Компилируем
supervisor_app = workflow.compile()

print("Supervisor граф готов!")

In [ ]:
# Визуализация
try:
    display(Image(supervisor_app.get_graph().draw_mermaid_png()))
except:
    print(supervisor_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
result = supervisor_app.invoke({
    "messages": [HumanMessage(content="Напиши про искусственный интеллект в медицине")],
    "next_worker": "",
    "research_data": "",
    "draft": "",
    "feedback": ""
})

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ:")
print("="*50)
print("\n📝 ЧЕРНОВИК:")
print(result["draft"])
print("\n💬 ФИДБЕК:")
print(result["feedback"])

### Преимущества Supervisor Pattern:

| Аспект | Handoff | Supervisor |
|--------|---------|------------|
| Гибкость | Жёсткий порядок | Динамический |
| Масштабирование | Сложно | Легко добавить агента |
| Контроль | Распределённый | Централизованный |
| Отладка | Сложнее | Проще (всё через boss) |

---

## 3. Subgraphs — Модульная архитектура

### Идея:

Граф внутри графа. Каждый subgraph — отдельный модуль со своей логикой.

```
Main Graph
├── Subgraph A (Research Team)
│   ├── Web Searcher
│   └── Summarizer
└── Subgraph B (Writing Team)
    ├── Drafter
    └── Editor
```

### Java аналогия:

Subgraph = Maven Module / Gradle Subproject:
- Изолированная логика
- Своя область ответственности
- Можно переиспользовать

In [ ]:
# === SUBGRAPH 1: Research Team ===

class ResearchState(TypedDict):
    query: str
    raw_data: str
    summary: str

def search_node(state: ResearchState) -> ResearchState:
    print("   🔎 Searcher: ищу информацию...")
    
    # Симуляция поиска
    response = llm.invoke([
        SystemMessage(content="Сгенерируй 5 фактов по теме. Формат: пункты."),
        HumanMessage(content=state["query"])
    ])
    
    return {"raw_data": response.content}

def summarize_node(state: ResearchState) -> ResearchState:
    print("   📋 Summarizer: обобщаю...")
    
    response = llm.invoke([
        SystemMessage(content="Сделай краткое саммари (2-3 предложения)."),
        HumanMessage(content=state["raw_data"])
    ])
    
    return {"summary": response.content}

# Собираем subgraph
research_workflow = StateGraph(ResearchState)
research_workflow.add_node("search", search_node)
research_workflow.add_node("summarize", summarize_node)
research_workflow.set_entry_point("search")
research_workflow.add_edge("search", "summarize")
research_workflow.add_edge("summarize", END)

research_subgraph = research_workflow.compile()

print("Research Subgraph готов!")

In [ ]:
# === SUBGRAPH 2: Writing Team ===

class WritingState(TypedDict):
    source_material: str
    draft: str
    final_text: str

def draft_node(state: WritingState) -> WritingState:
    print("   ✏️ Drafter: пишу черновик...")
    
    response = llm.invoke([
        SystemMessage(content="Напиши черновик статьи (2 абзаца) на основе материала."),
        HumanMessage(content=state["source_material"])
    ])
    
    return {"draft": response.content}

def edit_node(state: WritingState) -> WritingState:
    print("   🔧 Editor: редактирую...")
    
    response = llm.invoke([
        SystemMessage(content="Улучши текст: исправь ошибки, сделай более читаемым."),
        HumanMessage(content=state["draft"])
    ])
    
    return {"final_text": response.content}

# Собираем subgraph
writing_workflow = StateGraph(WritingState)
writing_workflow.add_node("draft", draft_node)
writing_workflow.add_node("edit", edit_node)
writing_workflow.set_entry_point("draft")
writing_workflow.add_edge("draft", "edit")
writing_workflow.add_edge("edit", END)

writing_subgraph = writing_workflow.compile()

print("Writing Subgraph готов!")

In [ ]:
# === MAIN GRAPH: объединяем subgraphs ===

class MainState(TypedDict):
    user_query: str
    research_summary: str
    final_article: str

def call_research_team(state: MainState) -> MainState:
    print("\n📚 RESEARCH TEAM:")
    
    # Вызываем subgraph
    result = research_subgraph.invoke({
        "query": state["user_query"],
        "raw_data": "",
        "summary": ""
    })
    
    return {"research_summary": result["summary"]}

def call_writing_team(state: MainState) -> MainState:
    print("\n✍️ WRITING TEAM:")
    
    # Вызываем subgraph
    result = writing_subgraph.invoke({
        "source_material": state["research_summary"],
        "draft": "",
        "final_text": ""
    })
    
    return {"final_article": result["final_text"]}

# Собираем main graph
main_workflow = StateGraph(MainState)
main_workflow.add_node("research_team", call_research_team)
main_workflow.add_node("writing_team", call_writing_team)
main_workflow.set_entry_point("research_team")
main_workflow.add_edge("research_team", "writing_team")
main_workflow.add_edge("writing_team", END)

main_app = main_workflow.compile()

print("Main Graph с subgraphs готов!")

In [ ]:
# Запускаем!
result = main_app.invoke({
    "user_query": "Блокчейн в банковской сфере",
    "research_summary": "",
    "final_article": ""
})

print("\n" + "="*50)
print("ФИНАЛЬНАЯ СТАТЬЯ:")
print("="*50)
print(result["final_article"])

### Преимущества Subgraphs:

| Преимущество | Описание |
|--------------|----------|
| **Модульность** | Каждый subgraph — отдельный модуль |
| **Переиспользование** | Research team можно использовать в других проектах |
| **Тестирование** | Можно тестировать subgraph отдельно |
| **Читаемость** | Код организован логически |
| **Масштабирование** | Легко добавлять новые команды |

---

## 4. Практика: Полноценная команда агентов

Создадим реальную систему:
- **Planner** — разбивает задачу на шаги
- **Executor** — выполняет шаги
- **Reviewer** — проверяет результат

In [ ]:
# Состояние для Plan-Execute-Review
class PlanExecuteState(TypedDict):
    task: str
    plan: list[str]
    current_step: int
    results: Annotated[list, operator.add]
    final_result: str
    review: str
    approved: bool

print("State готов!")

In [ ]:
import json

def planner_node(state: PlanExecuteState) -> PlanExecuteState:
    print("\n📋 PLANNER: создаю план...")
    
    response = llm.invoke([
        SystemMessage(content="""Разбей задачу на 3-5 конкретных шагов.
Ответь JSON массивом строк: ["шаг 1", "шаг 2", ...]
Только JSON, без объяснений."""),
        HumanMessage(content=state["task"])
    ])
    
    try:
        # Пытаемся распарсить JSON
        content = response.content.strip()
        # Убираем markdown если есть
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        plan = json.loads(content)
    except:
        # Fallback
        plan = ["Исследовать тему", "Написать черновик", "Отредактировать"]
    
    print(f"   План: {plan}")
    
    return {
        "plan": plan,
        "current_step": 0
    }

print("Planner создан!")

In [ ]:
def executor_node(state: PlanExecuteState) -> PlanExecuteState:
    step_idx = state["current_step"]
    current_step = state["plan"][step_idx]
    
    print(f"\n⚡ EXECUTOR: выполняю шаг {step_idx + 1}/{len(state['plan'])}")
    print(f"   Шаг: {current_step}")
    
    # Контекст предыдущих результатов
    prev_results = "\n".join(state.get("results", [])) if state.get("results") else "Нет"
    
    response = llm.invoke([
        SystemMessage(content=f"""Ты исполнитель. Выполни текущий шаг задачи.

Общая задача: {state['task']}
Текущий шаг: {current_step}
Предыдущие результаты: {prev_results}

Выполни шаг и верни результат. Кратко и по делу."""),
        HumanMessage(content=f"Выполни: {current_step}")
    ])
    
    result = f"[Шаг {step_idx + 1}] {current_step}: {response.content}"
    print(f"   Результат: {response.content[:100]}...")
    
    return {
        "results": [result],
        "current_step": step_idx + 1
    }

print("Executor создан!")

In [ ]:
def reviewer_node(state: PlanExecuteState) -> PlanExecuteState:
    print("\n🔍 REVIEWER: проверяю результаты...")
    
    all_results = "\n".join(state["results"])
    
    response = llm.invoke([
        SystemMessage(content="""Ты ревьюер. Проверь выполнение задачи.

Ответь в формате:
ОЦЕНКА: (1-10)
ВЕРДИКТ: APPROVED или NEEDS_REVISION
КОММЕНТАРИЙ: (краткий фидбек)"""),
        HumanMessage(content=f"""Задача: {state['task']}

Результаты:
{all_results}""")
    ])
    
    review_text = response.content
    approved = "APPROVED" in review_text.upper()
    
    print(f"   Вердикт: {'✅ APPROVED' if approved else '❌ NEEDS_REVISION'}")
    
    return {
        "review": review_text,
        "approved": approved,
        "final_result": all_results if approved else ""
    }

print("Reviewer создан!")

In [ ]:
# Роутеры
def should_continue_execution(state: PlanExecuteState) -> str:
    """Проверяем, есть ли ещё шаги."""
    if state["current_step"] < len(state["plan"]):
        return "executor"  # Ещё есть шаги
    return "reviewer"  # Все шаги выполнены

def after_review(state: PlanExecuteState) -> str:
    """После ревью — заканчиваем или переделываем."""
    if state["approved"]:
        return "__end__"
    # Можно вернуться к planner для переделки
    # Но для простоты — заканчиваем
    return "__end__"

print("Роутеры готовы!")

In [ ]:
# Собираем граф
per_workflow = StateGraph(PlanExecuteState)

# Узлы
per_workflow.add_node("planner", planner_node)
per_workflow.add_node("executor", executor_node)
per_workflow.add_node("reviewer", reviewer_node)

# Рёбра
per_workflow.set_entry_point("planner")
per_workflow.add_edge("planner", "executor")

per_workflow.add_conditional_edges(
    "executor",
    should_continue_execution,
    {
        "executor": "executor",
        "reviewer": "reviewer"
    }
)

per_workflow.add_conditional_edges(
    "reviewer",
    after_review,
    {
        "__end__": END
    }
)

# Компилируем с checkpointer
checkpointer = MemorySaver()
per_app = per_workflow.compile(checkpointer=checkpointer)

print("Plan-Execute-Review граф готов!")

In [ ]:
# Визуализация
try:
    display(Image(per_app.get_graph().draw_mermaid_png()))
except:
    print(per_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
config = {"configurable": {"thread_id": "per-demo-1"}}

result = per_app.invoke({
    "task": "Создай краткий гайд по Docker для Java-разработчика",
    "plan": [],
    "current_step": 0,
    "results": [],
    "final_result": "",
    "review": "",
    "approved": False
}, config=config)

print("\n" + "="*50)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print("="*50)
print("\n📝 РЕЗУЛЬТАТЫ ВЫПОЛНЕНИЯ:")
for r in result["results"]:
    print(f"\n{r}")
print("\n" + "-"*50)
print("📋 РЕВЬЮ:")
print(result["review"])

---

## Итоги

### Что изучили:

| Паттерн | Когда использовать | Сложность |
|---------|-------------------|------------|
| **Handoff** | Простая цепочка агентов | ⭐ |
| **Supervisor** | Динамическая маршрутизация | ⭐⭐ |
| **Subgraphs** | Модульная архитектура | ⭐⭐⭐ |
| **Plan-Execute** | Сложные многошаговые задачи | ⭐⭐⭐ |

### Java аналогии:

| LangGraph | Java/Spring |
|-----------|-------------|
| Supervisor | Orchestrator / Saga Coordinator |
| Workers | Microservices |
| Subgraphs | Maven Modules |
| State | Saga State / Event Store |

### Когда что использовать:

- **1 агент** — простые задачи, мало инструментов
- **Handoff** — фиксированный pipeline (A → B → C)
- **Supervisor** — динамический выбор агента
- **Subgraphs** — большие проекты, переиспользование

### Следующие шаги:

1. **LangGraph Cloud** — деплой в продакшен
2. **Память** — long-term memory для агентов
3. **Инструменты** — интеграция с реальными API